In [ ]:
import shutil, os
shutil.rmtree("/kaggle/working/calidrift_results")
os.makedirs("/kaggle/working/calidrift_results")
print("Clean")

In [4]:
# ============================================================
# CaliDrift: Complete Final Version
# Output: 2 files only
#   1. checkpoint.json     — resume state (delete to restart)
#   2. calidrift_results.xlsx — all paper tables in one Excel file
#      Sheet: RawData, Table5, Table6, Table7, Table8, Table9,
#             Table10, Figure5_RHR
# All fixes applied:
#   - make_prompt handles base/instruct separately
#   - VC_REGEX defined before use
#   - FaithDial flexible field detection
#   - Checkpoint survives crashes (atomic write)
#   - Analysis writes ONE Excel file, no separate CSVs
# ============================================================

import os, re, json, warnings, gc, subprocess
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm
from datetime import datetime

warnings.filterwarnings("ignore")

subprocess.run(["pip", "install", "statsmodels", "scikit-learn", "openpyxl", "-q"],
               capture_output=True)
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import roc_auc_score

# ============================================================
# CONFIGURATION
# ============================================================
SAMPLE_SIZE    = 5          # 1=test | 10=quick | 50=validate | 500=paper
SEEDS          = [1,2,3,4,5]
MAX_NEW_TOKENS = 128
RHR_THRESHOLD  = 0.60
MAX_LENGTH     = 512
N_BOOTSTRAP    = 10000      # set to 1000 for speed

MODEL_PAIRS = [
    {
        "pair_name":   "Qwen2.5-1.5B",
        "base_id":     "Qwen/Qwen2.5-1.5B",
        "instruct_id": "Qwen/Qwen2.5-1.5B-Instruct",
    },
    # Uncomment after adding HF_TOKEN secret in Kaggle:
    # {
    #     "pair_name":   "Gemma-2B",
    #     "base_id":     "google/gemma-2b",
    #     "instruct_id": "google/gemma-2b-it",
    # },
    # {
    #     "pair_name":   "Llama-3.2-1B",
    #     "base_id":     "meta-llama/Llama-3.2-1B",
    #     "instruct_id": "meta-llama/Llama-3.2-1B-Instruct",
    # },
]

# ============================================================
# PATHS — only 2 output files
# ============================================================
OUTDIR      = "/kaggle/working/calidrift_results"
CHECKPOINT  = f"{OUTDIR}/checkpoint.json"
EXCEL_OUT   = f"{OUTDIR}/calidrift_results.xlsx"
DATASET_SLUG = "datasets/samuelstephen77/calidrift-dataset"

os.makedirs(OUTDIR, exist_ok=True)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

print("="*60)
print("CALIDRIFT — FINAL VERSION")
print(f"Device:      {device.upper()}")
print(f"Sample size: {SAMPLE_SIZE}")
print(f"Seeds:       {SEEDS}")
print(f"Models:      {[p['pair_name'] for p in MODEL_PAIRS]}")
print(f"Checkpoint:  {CHECKPOINT}")
print(f"Excel out:   {EXCEL_OUT}")
print("="*60)

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("HuggingFace authenticated")
except Exception:
    print("No HF token — open-access models only")

# ============================================================
# CHECKPOINT SYSTEM
# ============================================================
def load_checkpoint():
    if not os.path.exists(CHECKPOINT):
        return set(), []
    try:
        with open(CHECKPOINT, "r") as f:
            ckpt = json.load(f)
        keys    = set(tuple(k) for k in ckpt.get("completed_keys", []))
        results = ckpt.get("results", [])
        print(f"Checkpoint loaded: {len(results)} results, {len(keys)} keys")
        return keys, results
    except Exception as e:
        print(f"Checkpoint read failed ({e})")
        # Fallback: recover from Excel if checkpoint corrupted
        if os.path.exists(EXCEL_OUT):
            try:
                recovered = pd.read_excel(EXCEL_OUT, sheet_name="RawData")\
                              .to_dict("records")
                keys = set(
                    (r["pair_name"], r["model_type"], r["dataset"],
                     r["seed"], r["sample_idx"])
                    for r in recovered
                    if all(k in r for k in
                           ["pair_name","model_type","dataset","seed","sample_idx"])
                )
                print(f"Recovered {len(recovered)} results from Excel")
                return keys, recovered
            except Exception as e2:
                print(f"Excel recovery failed ({e2}) — starting fresh")
        return set(), []

def save_checkpoint(completed_keys, results):
    tmp = CHECKPOINT + ".tmp"
    with open(tmp, "w") as f:
        json.dump({
            "completed_keys": [list(k) for k in completed_keys],
            "results":        results,
            "saved_at":       datetime.now().isoformat(),
            "n_results":      len(results),
        }, f)
    os.replace(tmp, CHECKPOINT)

# ============================================================
# HEDGE LEXICON
# ============================================================
HEDGE_PATTERNS = [
    r'\bmay\b', r'\bmight\b', r'\bcould\b', r'\bpossibly\b', r'\bperhaps\b',
    r'\bprobably\b', r'\blikely\b', r'\bseems?\b', r'\bappears?\b',
    r'\bit is (generally|widely|often) (believed|thought|understood|accepted)',
    r'\bsome evidence suggests?\b', r'\baccording to some\b',
    r'\bit has been suggested\b', r'\bit is possible that\b',
    r'\bone might argue\b', r'\bsome would say\b',
    r'\bhistorically\b', r'\bin (certain|some|many) contexts?\b',
    r'\bin (most|some|many) cases?\b', r'\bgenerally speaking\b',
    r'\bfor the most part\b', r'\bby and large\b',
    r'\bI (am not|\'m not) (entirely |completely )?sure\b',
    r'\bI (am|\'m) not certain\b', r'\bI (believe|think|suppose)\b',
    r'\bto (my|the best of my) knowledge\b',
    r'\bif I (recall|remember) correctly\b',
    r'\bit\'s worth noting\b', r'\bone caveat\b',
    r'\bapproximately\b', r'\broughly\b', r'\baround\b',
]
HEDGE_REGEX = re.compile('|'.join(HEDGE_PATTERNS), re.IGNORECASE)

# ============================================================
# PROMPT + PARSING
# VC_REGEX defined here before any use
# ============================================================
VC_REGEX = re.compile(r'CONFIDENCE:\s*(\d{1,3})', re.IGNORECASE)

def make_prompt(question, mtype="instruct"):
    """Separate prompt format for base vs instruct models."""
    if mtype == "base":
        # Simpler format — base models don't follow complex instructions
        return (
            f"Q: {question}\n"
            f"A: [write answer here]\n"
            f"CONFIDENCE: [write a number 0-100 here]\n"
            f"A:"
        )
    return (
        f"Question: {question}\n\n"
        "Answer the question above. Then on the next line write exactly:\n"
        "CONFIDENCE: [a number between 0 and 100]\n\n"
        "Answer:"
    )

def parse_vc(text):
    m = VC_REGEX.search(text)
    if m:
        val = int(m.group(1))
        if 0 <= val <= 100:
            return val / 100.0
    return None

# ============================================================
# DATASET LOADING
# SimpleQA: simple_qa_test_set.csv is at dataset root — no subfolder needed
# FaithDial: test.json at dataset root — flexible field detection
# ============================================================
def load_truthfulqa(n, seed):
    ds = load_dataset("truthful_qa", "generation", split="validation")
    ds = ds.shuffle(seed=seed).select(range(min(n, len(ds))))
    return [{"question": r["question"], "reference": r["best_answer"]} for r in ds]

def load_simpleqa(n, seed):
    # simple_qa_test_set.csv sits at the dataset root — no subfolder
    path = f"/kaggle/input/{DATASET_SLUG}/simple_qa_test_set.csv"
    try:
        df_sq = pd.read_csv(path)
        df_sq = df_sq.sample(frac=1, random_state=seed).reset_index(drop=True)
        items = [{"question": str(r["problem"]), "reference": str(r["answer"])}
                 for _, r in df_sq.head(n).iterrows()]
        print(f"    SimpleQA loaded: {len(items)} items")
        return items
    except Exception as e:
        print(f"    SimpleQA failed ({e}) — using TruthfulQA offset")
        ds = load_dataset("truthful_qa", "generation", split="validation")
        ds = ds.shuffle(seed=seed+100).select(range(min(n, len(ds))))
        return [{"question": r["question"], "reference": r["best_answer"]} for r in ds]

def load_faithdial(n, seed):
    path = f"/kaggle/input/{DATASET_SLUG}/test.json"
    try:
        with open(path) as f:
            data = json.load(f)
        items = []
        for record in data:
            # FaithDial structure: record → utterances list → each utterance has history, knowledge, response
            for utt in record.get("utterances", []):
                history   = utt.get("history", [])
                question  = history[-1] if history else ""
                knowledge = utt.get("knowledge", "")
                if question and knowledge:
                    items.append({
                        "question":  question,
                        "reference": knowledge
                    })
        rng = np.random.RandomState(seed)
        rng.shuffle(items)
        print(f"    FaithDial loaded: {len(items)} items")
        return items[:min(n, len(items))]
    except Exception as e:
        print(f"    FaithDial failed ({e}) — using TruthfulQA offset")
        ds = load_dataset("truthful_qa", "generation", split="validation")
        ds = ds.shuffle(seed=seed+200).select(range(min(n, len(ds))))
        return [{"question": r["question"], "reference": r["best_answer"]} for r in ds]

DATASET_LOADERS = {
    "TruthfulQA": load_truthfulqa,
    "SimpleQA":   load_simpleqa,
    "FaithDial":  load_faithdial,
}
DATASETS = {k: {"n_samples": SAMPLE_SIZE} for k in DATASET_LOADERS}

# ============================================================
# RESPONSE LABELING
# ============================================================
ABSTENTION_RE = re.compile(
    r"i (don'?t|do not) know|i('m| am) not sure|"
    r"i cannot (determine|answer|say)|i have no (information|knowledge)|"
    r"(unknown|unclear|not available|not specified)",
    re.IGNORECASE
)

def normalize(t):
    return re.sub(r'[^a-z0-9 ]', '', t.lower()).strip()

def label_response(response, reference):
    if ABSTENTION_RE.search(response):
        return "abstained"
    rn, fn = normalize(response), normalize(reference)
    if fn and fn in rn:
        return "correct"
    words = [w for w in fn.split() if len(w) > 3]
    if words and sum(w in rn.split() for w in words) >= max(1, len(words)//2):
        return "correct"
    return "hallucinated"

def is_rh(response, ic):
    return bool(HEDGE_REGEX.search(response)) and ic >= RHR_THRESHOLD

# ============================================================
# MODEL LOADING
# ============================================================
def load_model(model_id):
    print(f"    Loading {model_id}...")
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.truncation_side = "left"
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if "gemma" in model_id.lower() else torch.float16
    else:
        dtype = torch.float32
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,
        device_map={"": device}, trust_remote_code=True,
    )
    mdl.eval()
    return tok, mdl

# ============================================================
# GENERATION + ENTROPY
# ============================================================
def generate_with_entropy(tok, mdl, prompt):
    inp = tok(prompt, return_tensors="pt", truncation=True,
              max_length=MAX_LENGTH)
    inp = {k: v.to(mdl.device) for k, v in inp.items()}
    plen = inp["input_ids"].shape[1]

    with torch.no_grad():
        out = mdl.generate(
            **inp, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, temperature=1.0,
            return_dict_in_generate=True, output_scores=True,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )

    gen_ids = out.sequences[0][plen:]
    text    = tok.decode(gen_ids, skip_special_tokens=True)

    entropies = []
    for i, tid in enumerate(gen_ids.tolist()):
        if tid in (tok.eos_token_id, tok.pad_token_id):
            continue
        if i >= len(out.scores):
            break
        logits = out.scores[i][0].float()       # explicit float cast
        probs  = torch.softmax(logits, dim=-1)
        H_tok  = -(probs * torch.log(probs + 1e-12)).sum().item()
        entropies.append(H_tok)

    H  = float(np.mean(entropies)) if entropies else 0.0
    IC = float(np.exp(-H))
    return text, H, IC

# ============================================================
# MAIN EXPERIMENT LOOP
# ============================================================
completed_keys, all_results = load_checkpoint()
print(f"Starting with {len(all_results)} existing results")

SAVE_EVERY  = 25
total_runs  = len(MODEL_PAIRS) * len(DATASETS) * len(SEEDS)
run_n       = 0

for pair in MODEL_PAIRS:
    for ds_name in DATASETS:
        for seed in SEEDS:
            run_n += 1
            print(f"\n>>> Run {run_n}/{total_runs}: "
                  f"{pair['pair_name']} | {ds_name} | seed={seed}")

            items = DATASET_LOADERS[ds_name](
                DATASETS[ds_name]["n_samples"], seed)
            print(f"  Loaded {len(items)} samples")

            for mtype in ["base", "instruct"]:
                model_id   = pair[f"{mtype}_id"]
                model_name = model_id.split("/")[-1]

                remaining = [i for i in range(len(items))
                             if (pair["pair_name"],mtype,ds_name,seed,i)
                             not in completed_keys]

                if not remaining:
                    print(f"  {model_name}: already complete, skipping")
                    continue

                done_count = len([k for k in completed_keys
                                  if k[0]==pair["pair_name"] and k[1]==mtype
                                  and k[2]==ds_name and k[3]==seed])
                print(f"  {model_name}: {len(remaining)} remaining "
                      f"({done_count} already done)")

                tok, mdl   = load_model(model_id)
                new_since_save = 0

                for i in tqdm(remaining, desc=f"  {model_name}"):
                    # Use separate prompt format for base vs instruct
                    prompt = make_prompt(items[i]["question"], mtype)
                    try:
                        response, H, IC = generate_with_entropy(tok, mdl, prompt)
                    except Exception as e:
                        print(f"    item {i} error: {e}")
                        continue

                    VC = parse_vc(response)
                    key = (pair["pair_name"], mtype, ds_name, seed, i)
                    completed_keys.add(key)

                    if VC is None:
                        continue   # parse failure — mark done, skip result

                    all_results.append({
                        "pair_name":  pair["pair_name"],
                        "model_type": mtype,
                        "model_name": model_name,
                        "dataset":    ds_name,
                        "seed":       seed,
                        "sample_idx": i,
                        "VC":         round(VC, 6),
                        "IC":         round(IC, 6),
                        "H":          round(H, 6),
                        "CDI":        round(abs(VC - IC), 6),
                        "drift_mode": ("UED" if VC < IC else
                                       "OED" if VC > IC else "balanced"),
                        "label":      label_response(response,
                                                     items[i]["reference"]),
                        "rh":         is_rh(response, IC),
                        "response":   response[:600],
                    })
                    new_since_save += 1

                    if new_since_save >= SAVE_EVERY:
                        save_checkpoint(completed_keys, all_results)
                        new_since_save = 0

                save_checkpoint(completed_keys, all_results)
                print(f"  {model_name} done. Total results: {len(all_results)}")

                del mdl, tok
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()

# Final checkpoint save
save_checkpoint(completed_keys, all_results)
print(f"\nExperiment complete. Total results: {len(all_results)}")

# ============================================================
# ANALYSIS — ONE Excel file, multiple sheets
# ============================================================
if len(all_results) < 2:
    print("\nToo few results. Re-run with SAMPLE_SIZE >= 10.")
else:
    df   = pd.DataFrame(all_results)
    df_c = df.dropna(subset=["CDI","VC","IC"]).copy()
    PAIRS = df_c["pair_name"].unique().tolist()
    DSS   = df_c["dataset"].unique().tolist()

    print(f"\nAnalysing {len(df_c)} rows | Pairs: {PAIRS} | Datasets: {DSS}")

    # ── helpers ──────────────────────────────────────────────────────────
    def cohens_d(a, b):
        a, b = np.array(a), np.array(b)
        ps = np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
        return abs(a.mean()-b.mean()) / ps if ps > 0 else 0.0

    def bci(vals, n=N_BOOTSTRAP):
        v = np.array(vals, dtype=float); v = v[~np.isnan(v)]
        if len(v) == 0: return np.nan, np.nan
        m = [np.mean(np.random.choice(v,len(v),replace=True)) for _ in range(n)]
        return round(np.percentile(m,2.5),4), round(np.percentile(m,97.5),4)

    def ece(vc, correct, bins=10):
        vc = np.array(vc); c = np.array(correct, dtype=float)
        edges = np.linspace(0,1,bins+1); out = 0.0
        for i in range(bins):
            hi = edges[i+1] if i < bins-1 else edges[i+1]+1e-9
            mask = (vc >= edges[i]) & (vc < hi)
            if mask.sum() == 0: continue
            out += mask.mean() * abs(c[mask].mean() - vc[mask].mean())
        return round(out, 4)

    def dom_drift(sub_df):
        d = sub_df["drift_mode"].value_counts()
        return d.index[0] if len(d) > 0 else "N/A"

    sheets = {}  # sheet_name → DataFrame

    # ── RawData sheet ─────────────────────────────────────────────────────
    sheets["RawData"] = df.drop(columns=["response"], errors="ignore")

    # ── Table5 ────────────────────────────────────────────────────────────
    t5, t5_pv = [], []
    for pname in PAIRS:
        p  = df_c[df_c["pair_name"]==pname]
        bc = p[p["model_type"]=="base"]["CDI"].values
        ic = p[p["model_type"]=="instruct"]["CDI"].values
        if len(bc)==0 or len(ic)==0: continue
        bsm = p[p["model_type"]=="base"].groupby("seed")["CDI"].mean().values
        ism = p[p["model_type"]=="instruct"].groupby("seed")["CDI"].mean().values
        n   = min(len(bsm), len(ism))
        _, pv = (stats.ttest_rel(ism[:n], bsm[:n]) if n >= 2
                 else stats.ttest_ind(ic, bc))
        t5_pv.append(pv)
        raw_p = df[df["pair_name"]==pname]
        pfr_b = (len(raw_p[raw_p["model_type"]=="base"]) - len(bc)) / \
                max(len(raw_p[raw_p["model_type"]=="base"]), 1) * 100
        pfr_i = (len(raw_p[raw_p["model_type"]=="instruct"]) - len(ic)) / \
                max(len(raw_p[raw_p["model_type"]=="instruct"]), 1) * 100
        blo, bhi = bci(bc); ilo, ihi = bci(ic)
        t5.append({
            "Pair":          pname,
            "CDI_base_mean": round(bc.mean(), 3),
            "CDI_base_sd":   round(bc.std(ddof=1), 3),
            "CDI_base_95CI": f"[{blo},{bhi}]",
            "CDI_inst_mean": round(ic.mean(), 3),
            "CDI_inst_sd":   round(ic.std(ddof=1), 3),
            "CDI_inst_95CI": f"[{ilo},{ihi}]",
            "delta_CDI":     round(ic.mean()-bc.mean(), 3),
            "p_raw":         round(pv, 4),
            "cohen_d":       round(cohens_d(ic, bc), 2),
            "dom_drift":     dom_drift(p[p["model_type"]=="instruct"]),
            "PFR_base_pct":  round(pfr_b, 1),
            "PFR_inst_pct":  round(pfr_i, 1),
        })
        print(f"T5 {pname}: Δ={ic.mean()-bc.mean():+.3f} p={pv:.4f} d={cohens_d(ic,bc):.2f}")

    if len(t5_pv) >= 2:
        _, pc, _, _ = multipletests(t5_pv, method="holm")
        for i, r in enumerate(t5): r["p_corrected"] = round(pc[i], 4)
        print(f"Holm-Bonferroni p (T5): {[round(p,4) for p in pc]}")
    else:
        for r in t5: r["p_corrected"] = r["p_raw"]
    sheets["Table5_CDI_by_Pair"] = pd.DataFrame(t5)

    # ── Table6 ────────────────────────────────────────────────────────────
    t6 = []
    for ds in DSS:
        d  = df_c[df_c["dataset"]==ds]
        bc = d[d["model_type"]=="base"]["CDI"].values
        ic = d[d["model_type"]=="instruct"]["CDI"].values
        ece_b = ece(d[d["model_type"]=="base"]["VC"].values,
                    (d[d["model_type"]=="base"]["label"]=="correct").astype(int).values)
        ece_i = ece(d[d["model_type"]=="instruct"]["VC"].values,
                    (d[d["model_type"]=="instruct"]["label"]=="correct").astype(int).values)
        t6.append({
            "Dataset":       ds,
            "CDI_base_mean": round(bc.mean(), 3) if len(bc) > 0 else np.nan,
            "CDI_base_sd":   round(bc.std(ddof=1), 3) if len(bc) > 1 else np.nan,
            "CDI_inst_mean": round(ic.mean(), 3) if len(ic) > 0 else np.nan,
            "CDI_inst_sd":   round(ic.std(ddof=1), 3) if len(ic) > 1 else np.nan,
            "delta_CDI":     round(ic.mean()-bc.mean(), 3) if len(bc)>0 and len(ic)>0 else np.nan,
            "ECE_base":      ece_b,
            "ECE_inst":      ece_i,
            "dom_drift":     dom_drift(d[d["model_type"]=="instruct"]),
        })
        print(f"T6 {ds}: ECE_b={ece_b:.3f} ECE_i={ece_i:.3f}")
    sheets["Table6_CDI_by_Dataset"] = pd.DataFrame(t6)

    # ── Table7 ────────────────────────────────────────────────────────────
    t7, t7_pv = [], []
    inst = df_c[df_c["model_type"]=="instruct"]
    for ds in DSS:
        sub  = inst[inst["dataset"]==ds]
        corr = sub[sub["label"]=="correct"]["CDI"].values
        hall = sub[sub["label"]=="hallucinated"]["CDI"].values
        if len(corr) < 3 or len(hall) < 3:
            print(f"T7 {ds}: skipped (corr={len(corr)}, hall={len(hall)})")
            continue
        _, pv = stats.ttest_ind(hall, corr)
        t7_pv.append(pv)
        t7.append({
            "Dataset":       ds,
            "CDI_corr_mean": round(corr.mean(), 3),
            "CDI_corr_sd":   round(corr.std(ddof=1), 3),
            "CDI_hall_mean": round(hall.mean(), 3),
            "CDI_hall_sd":   round(hall.std(ddof=1), 3),
            "delta":         round(hall.mean()-corr.mean(), 3),
            "p_raw":         round(pv, 4),
            "cohen_d":       round(cohens_d(hall, corr), 2),
            "dom_drift":     dom_drift(sub[sub["label"]=="hallucinated"]),
            "n_corr":        len(corr),
            "n_hall":        len(hall),
        })
        print(f"T7 {ds}: Δ={hall.mean()-corr.mean():+.3f} d={cohens_d(hall,corr):.2f}")

    if len(t7_pv) >= 2:
        _, pc7, _, _ = multipletests(t7_pv, method="holm")
        for i, r in enumerate(t7): r["p_corrected"] = round(pc7[i], 4)
        print(f"Holm-Bonferroni p (T7): {[round(p,4) for p in pc7]}")
    else:
        for r in t7: r["p_corrected"] = r.get("p_raw", np.nan)
    sheets["Table7_CDI_by_Label"] = pd.DataFrame(t7) if t7 else pd.DataFrame(
        columns=["Dataset","CDI_corr_mean","CDI_hall_mean","delta","p_raw","cohen_d"])

    # ── Table8 ────────────────────────────────────────────────────────────
    t8 = []
    for pname in PAIRS:
        p = df[df["pair_name"]==pname]
        for mtype in ["base","instruct"]:
            sub = p[p["model_type"]==mtype]
            if len(sub) == 0: continue
            t8.append({
                "Pair":      pname,
                "model_type":mtype,
                "RHR_mean":  round(sub["rh"].mean(), 3),
                "RHR_sd":    round(sub["rh"].std(ddof=1), 3) if len(sub)>1 else np.nan,
                "n":         len(sub),
            })
        bm = p[p["model_type"]=="base"]["rh"].mean()
        im = p[p["model_type"]=="instruct"]["rh"].mean()
        print(f"T8 {pname}: base={bm:.3f} inst={im:.3f} ΔRHR={im-bm:+.3f}")
    sheets["Table8_RHR"] = pd.DataFrame(t8)

    # ── Table9 ────────────────────────────────────────────────────────────
    t9 = []
    for pname in PAIRS:
        for mtype in ["base","instruct"]:
            sub = df_c[(df_c["pair_name"]==pname)&(df_c["model_type"]==mtype)]
            if len(sub) == 0: continue
            mname  = sub["model_name"].iloc[0] if "model_name" in sub.columns \
                     else f"{pname}-{mtype}"
            rhr_v  = df[(df["pair_name"]==pname)&(df["model_type"]==mtype)]["rh"].mean()
            ece_v  = ece(sub["VC"].values,
                         (sub["label"]=="correct").astype(int).values)
            t9.append({
                "Model":  mname, "Type": mtype, "Pair": pname,
                "CDI":    round(sub["CDI"].mean(), 3),
                "CDI_sd": round(sub["CDI"].std(ddof=1), 3) if len(sub)>1 else np.nan,
                "ECE":    ece_v,
                "RHR":    round(rhr_v, 3),
                "n":      len(sub),
            })
            print(f"T9 {mname}: CDI={sub['CDI'].mean():.3f} ECE={ece_v:.3f} RHR={rhr_v:.3f}")
    sheets["Table9_Cross_Model"] = pd.DataFrame(t9).sort_values("CDI", ascending=False)

    # ── Table10 ───────────────────────────────────────────────────────────
    inst_full = df_c[
        (df_c["model_type"]=="instruct") &
        (df_c["label"].isin(["correct","hallucinated"]))
    ].copy()
    t10 = []
    for ds in DSS:
        sub = inst_full[inst_full["dataset"]==ds]
        if len(sub) < 10:
            print(f"T10 {ds}: skipped (n={len(sub)}) — need >= 50 samples")
            continue
        labels = (sub["label"]=="hallucinated").astype(int).values
        if labels.sum()==0 or labels.sum()==len(labels):
            print(f"T10 {ds}: only one class")
            continue
        ece_sig = np.abs(sub["VC"].values - 0.5)
        try:
            r = {
                "Dataset":  ds,
                "VC_only":  round(roc_auc_score(labels, sub["VC"].values), 3),
                "IC_only":  round(roc_auc_score(labels, sub["IC"].values), 3),
                "ECE_only": round(roc_auc_score(labels, ece_sig), 3),
                "CDI":      round(roc_auc_score(labels, sub["CDI"].values), 3),
                "n_corr":   int((labels==0).sum()),
                "n_hall":   int((labels==1).sum()),
            }
            r["CDI_vs_best"] = round(
                r["CDI"] - max(r["VC_only"],r["IC_only"],r["ECE_only"]), 3)
            t10.append(r)
            print(f"T10 {ds}: CDI={r['CDI']} vs best_baseline={max(r['VC_only'],r['IC_only'],r['ECE_only'])}")
        except Exception as e:
            print(f"T10 {ds}: error — {e}")

    if t10:
        adf = pd.DataFrame(t10)
        avg = adf[["VC_only","IC_only","ECE_only","CDI"]].mean()
        print(f"T10 Average: VC={avg.VC_only:.3f} IC={avg.IC_only:.3f} "
              f"ECE={avg.ECE_only:.3f} CDI={avg.CDI:.3f}")
        sheets["Table10_Ablation"] = adf
    else:
        print("Table10 empty — run with SAMPLE_SIZE >= 50")
        sheets["Table10_Ablation"] = pd.DataFrame(
            columns=["Dataset","VC_only","IC_only","ECE_only","CDI","CDI_vs_best"])

    # ── Figure5 RHR Sensitivity ───────────────────────────────────────────
    thresholds = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
    sens = []
    for pname in PAIRS:
        p = df[df["pair_name"]==pname].copy()
        for tau in thresholds:
            for mtype in ["base","instruct"]:
                sub = p[p["model_type"]==mtype]
                if len(sub) == 0: continue
                rhr_tau = sub.apply(
                    lambda r: bool(HEDGE_REGEX.search(str(r.get("response",""))))
                              and float(r.get("IC", 0)) >= tau, axis=1).mean()
                sens.append({"pair":pname,"model_type":mtype,
                             "tau":tau,"RHR":round(rhr_tau,4)})
    sheets["Figure5_RHR_Sensitivity"] = pd.DataFrame(sens)

    # ── Write ONE Excel file ──────────────────────────────────────────────
    with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl") as writer:
        for sheet_name, sheet_df in sheets.items():
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\n{'='*60}")
    print(f"DONE — 2 files generated:")
    print(f"  {CHECKPOINT}")
    print(f"    Size: {os.path.getsize(CHECKPOINT)/1024:.1f} KB")
    print(f"  {EXCEL_OUT}")
    print(f"    Size: {os.path.getsize(EXCEL_OUT)/1024:.1f} KB")
    print(f"{'='*60}")
    print(f"""
Excel sheets:
  RawData                → all VC, IC, CDI per response
  Table5_CDI_by_Pair     → Table 5 (paper)
  Table6_CDI_by_Dataset  → Table 6 (paper)
  Table7_CDI_by_Label    → Table 7 (paper)
  Table8_RHR             → Table 8 (paper)
  Table9_Cross_Model     → Table 9 (paper)
  Table10_Ablation       → Table 10 (paper)
  Figure5_RHR_Sensitivity→ Figure 5 data (paper)

Delete checkpoint.json to restart from scratch.
""")


CALIDRIFT — FINAL VERSION
Device:      CPU
Sample size: 5
Seeds:       [1, 2, 3, 4, 5]
Models:      ['Qwen2.5-1.5B']
Checkpoint:  /kaggle/working/calidrift_results/checkpoint.json
Excel out:   /kaggle/working/calidrift_results/calidrift_results.xlsx
No HF token — open-access models only
Starting with 0 existing results

>>> Run 1/15: Qwen2.5-1.5B | TruthfulQA | seed=1


README.md: 0.00B [00:00, ?B/s]

generation/validation-00000-of-00001.par(…):   0%|          | 0.00/223k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

  Loaded 5 samples
  Qwen2.5-1.5B: 5 remaining (0 already done)
    Loading Qwen/Qwen2.5-1.5B...


...

  Qwen2.5-1.5B-Instruct: 100%|██████████| 5/5 [03:17<00:00, 39.55s/it]


  Qwen2.5-1.5B-Instruct done. Total results: 5

>>> Run 2/15: Qwen2.5-1.5B | TruthfulQA | seed=2
  Loaded 5 samples
  Qwen2.5-1.5B: 5 remaining (0 already done)
    Loading Qwen/Qwen2.5-1.5B...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Qwen2.5-1.5B:  60%|██████    | 3/5 [01:30<01:00, 30.15s/it]


KeyboardInterrupt: 

In [3]:
# ============================================================
# CALIDRIFT: Semantic Entropy Comparison
# Run this as a SEPARATE cell after main experiment is done
# Purpose: Compare IC (token entropy) vs Semantic Entropy
#          to validate CDI trend holds with stronger IC proxy
#
# Config: Gemma-2B-IT only, TruthfulQA only, 100 questions
# Time:   ~2-3 hours on T4 x2
# Output: semantic_entropy_results.json
# ============================================================

import os, json, re, gc, random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from datetime import datetime

# ── Config ────────────────────────────────────────────────────
SE_MODEL_ID   = "google/gemma-2b-it"
SE_DATASET    = "/kaggle/input/datasets/samuelstephen77/calidrift-dataset/TruthfulQA.csv"
SE_N_SAMPLES  = 100
SE_N_DRAWS    = 5        # number of sampled responses per question
SE_TEMP       = 0.7      # temperature for sampling
SE_MAX_TOKENS = 64
SE_SEED       = 42
SE_OUT        = "/kaggle/working/semantic_entropy_results.json"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")
print(f"Model: {SE_MODEL_ID}")
print(f"Samples: {SE_N_SAMPLES} questions × {SE_N_DRAWS} draws = {SE_N_SAMPLES * SE_N_DRAWS} generations")
print(f"Estimated time: ~{SE_N_SAMPLES * SE_N_DRAWS * 2 // 60} minutes")

# ── HF Auth ───────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("HuggingFace authenticated")
except Exception as e:
    print(f"HF auth: {e}")

# ── Load Dataset ──────────────────────────────────────────────
df = pd.read_csv(SE_DATASET)
df.columns = [c.strip() for c in df.columns]
q_col = next(c for c in df.columns if "question" in c.lower())
a_col = next(c for c in df.columns if "best" in c.lower() or "answer" in c.lower())
df = df.sample(frac=1, random_state=SE_SEED).head(SE_N_SAMPLES)
questions = [{"question": str(r[q_col]), "reference": str(r[a_col])} for _, r in df.iterrows()]
print(f"\nLoaded {len(questions)} questions from TruthfulQA")

# ── Load Model ────────────────────────────────────────────────
print(f"\nLoading {SE_MODEL_ID}...")
tok = AutoTokenizer.from_pretrained(SE_MODEL_ID, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

mdl = AutoModelForCausalLM.from_pretrained(
    SE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": device},
    trust_remote_code=True,
)
mdl.eval()
print(f"VRAM used: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

# ── Prompt ────────────────────────────────────────────────────
VC_REGEX = re.compile(r'CONFIDENCE:\s*(\d{1,3})', re.IGNORECASE)
VC_ALT   = re.compile(r'[Cc]onfidence[:\s]+(\d{1,3})')

def make_prompt(question):
    return (
        f"Question: {question}\n\n"
        "Answer the question. Then write:\n"
        "CONFIDENCE: [0-100]\n\nAnswer:"
    )

def parse_vc(text):
    m = VC_REGEX.search(text)
    if m:
        v = int(m.group(1))
        if 0 <= v <= 100: return v / 100.0
    m = VC_ALT.search(text)
    if m:
        v = int(m.group(1))
        if 0 <= v <= 100: return v / 100.0
    return None

# ── Normalize response for semantic clustering ─────────────────
def normalize_response(text):
    """Strip confidence field and normalize for semantic comparison"""
    # Remove everything after CONFIDENCE:
    text = re.sub(r'CONFIDENCE:.*', '', text, flags=re.IGNORECASE).strip()
    # Lowercase, remove punctuation
    text = re.sub(r'[^a-z0-9 ]', '', text.lower()).strip()
    return text

# ── Simple semantic clustering using string overlap ────────────
# (avoids needing a separate embedding model)
def semantic_cluster(responses):
    """
    Cluster responses by meaning using token overlap.
    Returns cluster assignments and cluster count.
    Simple but effective for short factual answers.
    """
    normalized = [normalize_response(r) for r in responses]
    clusters   = []
    cluster_id = 0

    assignments = [-1] * len(normalized)

    for i, ni in enumerate(normalized):
        if assignments[i] >= 0:
            continue
        assignments[i] = cluster_id
        tokens_i = set(ni.split())
        for j in range(i+1, len(normalized)):
            if assignments[j] >= 0:
                continue
            tokens_j = set(normalized[j].split())
            if not tokens_i or not tokens_j:
                continue
            # Jaccard similarity
            overlap = len(tokens_i & tokens_j) / len(tokens_i | tokens_j)
            if overlap >= 0.5:  # 50% token overlap = same cluster
                assignments[j] = cluster_id
        cluster_id += 1

    n_clusters = len(set(assignments))
    return assignments, n_clusters

# ── Generation with both IC and SE ────────────────────────────
def generate_single(prompt, do_sample=False, temperature=1.0):
    """Single generation returning text, IC"""
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=512)
    inp = {k: v.to(mdl.device) for k, v in inp.items()}
    plen = inp["input_ids"].shape[1]

    with torch.no_grad():
        out = mdl.generate(
            **inp,
            max_new_tokens=SE_MAX_TOKENS,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            repetition_penalty=1.1,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )

    gen_ids = out.sequences[0][plen:]
    text = tok.decode(gen_ids, skip_special_tokens=True).strip()

    entropies = []
    for i, tid in enumerate(gen_ids.tolist()):
        if tid in (tok.eos_token_id, tok.pad_token_id): continue
        if i >= len(out.scores): break
        probs = torch.softmax(out.scores[i][0].float(), dim=-1)
        H_tok = -(probs * torch.log(probs + 1e-12)).sum().item()
        entropies.append(H_tok)

    H  = float(np.mean(entropies)) if entropies else 0.0
    IC = float(np.exp(-H))
    return text, H, IC

# ── Main Loop ─────────────────────────────────────────────────
results = []
print(f"\n{'='*60}")
print("RUNNING SEMANTIC ENTROPY COMPARISON")
print(f"{'='*60}\n")

for idx, item in enumerate(questions):
    prompt = make_prompt(item["question"])

    # 1. Greedy generation — for IC and VC (same as main experiment)
    greedy_text, greedy_H, greedy_IC = generate_single(prompt, do_sample=False)
    greedy_VC = parse_vc(greedy_text)

    # 2. Sampled generations — for semantic entropy
    sampled_texts = []
    for _ in range(SE_N_DRAWS):
        text, _, _ = generate_single(prompt, do_sample=True, temperature=SE_TEMP)
        sampled_texts.append(text)

    # 3. Compute semantic entropy
    # Count distinct semantic clusters across sampled responses
    _, n_clusters = semantic_cluster(sampled_texts)

    # Semantic entropy = log(n_distinct_clusters)
    # Normalized to [0,1] using log(SE_N_DRAWS) as max
    SE_raw = np.log(n_clusters + 1e-9)
    SE_max = np.log(SE_N_DRAWS)
    SE_norm = float(SE_raw / SE_max) if SE_max > 0 else 0.0
    SE_norm = max(0.0, min(1.0, SE_norm))

    # Semantic IC = 1 - SE_norm (high semantic entropy = low confidence)
    Semantic_IC = 1.0 - SE_norm

    # 4. CDI variants
    CDI_token    = abs(greedy_VC - greedy_IC) if greedy_VC is not None else None
    CDI_semantic = abs(greedy_VC - Semantic_IC) if greedy_VC is not None else None

    results.append({
        "question":     item["question"][:100],
        "reference":    item["reference"][:100],
        "greedy_text":  greedy_text[:300],
        "VC":           round(greedy_VC, 4) if greedy_VC is not None else None,
        "H_token":      round(greedy_H, 4),
        "IC_token":     round(greedy_IC, 4),
        "n_clusters":   n_clusters,
        "SE_norm":      round(SE_norm, 4),
        "Semantic_IC":  round(Semantic_IC, 4),
        "CDI_token":    round(CDI_token, 4) if CDI_token is not None else None,
        "CDI_semantic": round(CDI_semantic, 4) if CDI_semantic is not None else None,
        "parse_ok":     greedy_VC is not None,
    })

    if (idx+1) % 10 == 0:
        parsed = sum(1 for r in results if r["parse_ok"])
        print(f"  [{idx+1}/{SE_N_SAMPLES}] parsed={parsed}/{idx+1} | "
              f"CDI_token={np.mean([r['CDI_token'] for r in results if r['CDI_token'] is not None]):.3f} | "
              f"CDI_sem={np.mean([r['CDI_semantic'] for r in results if r['CDI_semantic'] is not None]):.3f}")

# ── Save ──────────────────────────────────────────────────────
with open(SE_OUT, "w") as f:
    json.dump({
        "results":    results,
        "n_results":  len(results),
        "model":      SE_MODEL_ID,
        "dataset":    "TruthfulQA",
        "n_samples":  SE_N_SAMPLES,
        "n_draws":    SE_N_DRAWS,
        "temperature":SE_TEMP,
        "saved_at":   datetime.now().isoformat(),
    }, f)
print(f"\nSaved: {SE_OUT}")

# ── Summary ───────────────────────────────────────────────────
parsed = [r for r in results if r["parse_ok"]]
print(f"\n{'='*60}")
print("SEMANTIC ENTROPY COMPARISON SUMMARY")
print(f"{'='*60}")
print(f"Total:       {len(results)}")
print(f"Parsed:      {len(parsed)} ({len(parsed)/len(results)*100:.1f}%)")

if parsed:
    cdi_t = [r["CDI_token"] for r in parsed]
    cdi_s = [r["CDI_semantic"] for r in parsed]
    ic_t  = [r["IC_token"] for r in parsed]
    ic_s  = [r["Semantic_IC"] for r in parsed]
    vc    = [r["VC"] for r in parsed]

    print(f"\nMean VC:           {np.mean(vc):.3f}")
    print(f"Mean IC (token):   {np.mean(ic_t):.3f}")
    print(f"Mean IC (semantic):{np.mean(ic_s):.3f}")
    print(f"Mean CDI (token):  {np.mean(cdi_t):.3f} ± {np.std(cdi_t):.3f}")
    print(f"Mean CDI (semantic):{np.mean(cdi_s):.3f} ± {np.std(cdi_s):.3f}")

    # Correlation between two CDI measures
    corr = np.corrcoef(cdi_t, cdi_s)[0,1]
    print(f"\nCorrelation CDI_token vs CDI_semantic: r = {corr:.3f}")
    if corr >= 0.70:
        print("✅ HIGH correlation — token entropy CDI is a valid proxy")
        print("   Paper finding: CDI trend robust to IC proxy choice")
    elif corr >= 0.50:
        print("⚠️  MODERATE correlation — token entropy is acceptable proxy")
    else:
        print("❌ LOW correlation — token entropy is a weak proxy")
        print("   Consider using semantic IC as primary metric")

    # OED rate with both measures
    oed_t = sum(1 for r in parsed if r["VC"] > r["IC_token"]) / len(parsed)
    oed_s = sum(1 for r in parsed if r["VC"] > r["Semantic_IC"]) / len(parsed)
    print(f"\nOED rate (token entropy):    {oed_t*100:.1f}%")
    print(f"OED rate (semantic entropy): {oed_s*100:.1f}%")

print(f"{'='*60}")
print(f"Download: {SE_OUT}")
print("Then share semantic_entropy_results.json for paper update.")

del mdl, tok
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

Device: CUDA
Model: google/gemma-2b-it
Samples: 100 questions × 5 draws = 500 generations
Estimated time: ~16 minutes
HuggingFace authenticated

Loaded 100 questions from TruthfulQA

Loading google/gemma-2b-it...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

VRAM used: 5.01 GB

RUNNING SEMANTIC ENTROPY COMPARISON

  [10/100] parsed=8/10 | CDI_token=0.321 | CDI_sem=0.662
  [20/100] parsed=11/20 | CDI_token=0.364 | CDI_sem=0.724
  [30/100] parsed=17/30 | CDI_token=0.338 | CDI_sem=0.623
  [40/100] parsed=23/40 | CDI_token=0.346 | CDI_sem=0.618
  [50/100] parsed=31/50 | CDI_token=0.374 | CDI_sem=0.602
  [60/100] parsed=35/60 | CDI_token=0.385 | CDI_sem=0.635
  [70/100] parsed=42/70 | CDI_token=0.361 | CDI_sem=0.625
  [80/100] parsed=46/80 | CDI_token=0.368 | CDI_sem=0.647
  [90/100] parsed=51/90 | CDI_token=0.366 | CDI_sem=0.644
  [100/100] parsed=58/100 | CDI_token=0.376 | CDI_sem=0.658

Saved: /kaggle/working/semantic_entropy_results.json

SEMANTIC ENTROPY COMPARISON SUMMARY
Total:       100
Parsed:      58 (58.0%)

Mean VC:           0.850
Mean IC (token):   0.581
Mean IC (semantic):0.247
Mean CDI (token):  0.376 ± 0.148
Mean CDI (semantic):0.658 ± 0.300

Correlation CDI_token vs CDI_semantic: r = 0.329
❌ LOW correlation — token entropy is 

3611